# Netop 5G Network Energy Consumption - Data Preprocessing

**Phase 1:** Data Acquisition, Inspection, Cleaning, and Feature Engineering

## Overview

This notebook preprocesses three datasets:
- **BSinfo.csv**: Base station metadata (1,217 base stations)
- **CLstat.csv**: Cell-level load statistics (~125K hourly records)
- **ECstat.csv**: Energy consumption data (~92K hourly records)

## Steps:
1. Load raw data from `raw_data/` directory
2. Inspect each dataset for quality issues
3. Clean data (handle missing values, invalid entries)
4. Engineer features (time-based, lagged, rolling statistics)
5. Merge datasets: CLstat + BSinfo + ECstat
6. Save processed data to `processed_data/` directory

---
## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("✓ Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

---
## 2. Data Loading

Load the three CSV files from the `raw_data/` directory.

In [ ]:
# Load the three CSV files
bsinfo = pd.read_csv('raw_data/BSinfo.csv')
clstat = pd.read_csv('raw_data/CLstat.csv')
ecstat = pd.read_csv('raw_data/ECstat.csv')

print("="*80)
print("DATA LOADED SUCCESSFULLY")
print("="*80)
print(f"✓ BSinfo: {bsinfo.shape[0]:,} rows × {bsinfo.shape[1]} columns")
print(f"✓ CLstat: {clstat.shape[0]:,} rows × {clstat.shape[1]} columns")
print(f"✓ ECstat: {ecstat.shape[0]:,} rows × {ecstat.shape[1]} columns")

---
## 3. Data Inspection: BSinfo (Base Station Information)

Static metadata about base stations including hardware type, TX power, bandwidth, etc.

In [ ]:
print("="*80)
print("BSinfo - Base Station Information")
print("="*80)

print("\n📊 First 5 rows:")
display(bsinfo.head())

print("\n📋 Data types:")
display(bsinfo.dtypes)

print("\n❓ Missing values:")
display(bsinfo.isnull().sum())

print("\n📈 Basic statistics:")
display(bsinfo.describe())

print("\n🔢 Unique values:")
print(f"  - Base Stations: {bsinfo['BS'].nunique()}")
print(f"  - RU Types: {bsinfo['RUType'].nunique()}")
print(f"  - Modes: {bsinfo['Mode'].nunique()}")
print(f"\n  RU Type distribution:")
display(bsinfo['RUType'].value_counts())

---
## 4. Data Inspection: CLstat (Cell-Level Statistics)

Hourly load statistics for each cell, including traffic patterns and energy saving modes.

In [ ]:
print("="*80)
print("CLstat - Cell-Level Statistics")
print("="*80)

print("\n📊 First 5 rows:")
display(clstat.head())

print("\n📋 Data types:")
display(clstat.dtypes)

print("\n❓ Missing values:")
missing = clstat.isnull().sum()
display(missing[missing > 0])

print("\n📈 Load statistics:")
display(clstat['load'].describe())

print(f"\n📅 Date range: {clstat['Time'].min()} to {clstat['Time'].max()}")
print(f"\n🔢 Unique base stations: {clstat['BS'].nunique()}")
print(f"🔢 Unique cells: {clstat['CellName'].nunique()}")

---
## 5. Data Inspection: ECstat (Energy Consumption)

Hourly energy consumption measurements per base station.

In [ ]:
print("="*80)
print("ECstat - Energy Consumption")
print("="*80)

print("\n📊 First 5 rows:")
display(ecstat.head())

print("\n📋 Data types:")
display(ecstat.dtypes)

print("\n❓ Missing values:")
display(ecstat.isnull().sum())

print("\n📈 Energy statistics:")
display(ecstat['Energy'].describe())

print(f"\n📅 Date range: {ecstat['Time'].min()} to {ecstat['Time'].max()}")
print(f"\n🔢 Unique base stations: {ecstat['BS'].nunique()}")
print(f"🔢 Unique (Time, BS) combinations: {ecstat[['Time', 'BS']].drop_duplicates().shape[0]:,}")

---
## 6. PRE-CLEANING ANALYSIS: Missing Value Patterns

**CRITICAL:** Before deciding on a missing value strategy, let's analyze:
1. How many missing values exist?
2. Which columns have missing values?
3. Are they random or systematic?
4. What's the temporal pattern?

This will help us choose between:
- **ffill/bfill** (current approach)
- **Time-aware interpolation** (better for time series)
- **Skip filling** (if no missing values exist)
- **Domain-specific logic** (if systematic patterns)

In [ ]:
print("="*80)
print("PRE-CLEANING INSPECTION: Missing Value Analysis")
print("="*80)

# ============================================================================
# PART 1: Overall Missing Value Counts
# ============================================================================
print("\n" + "="*80)
print("PART 1: Overall Missing Value Counts")
print("="*80)

print("\n📊 BSinfo missing values:")
missing_bs = bsinfo.isnull().sum()
if missing_bs.sum() > 0:
    display(missing_bs[missing_bs > 0])
    print(f"  Total: {missing_bs.sum()} missing values ({100*missing_bs.sum()/(bsinfo.shape[0]*bsinfo.shape[1]):.2f}% of all cells)")
else:
    print("  ✅ No missing values in BSinfo!")

print("\n📊 CLstat missing values:")
missing_cl = clstat.isnull().sum()
if missing_cl.sum() > 0:
    display(missing_cl[missing_cl > 0])
    print(f"  Total: {missing_cl.sum()} missing values ({100*missing_cl.sum()/(clstat.shape[0]*clstat.shape[1]):.2f}% of all cells)")
else:
    print("  ✅ No missing values in CLstat!")

print("\n⚡ ECstat missing values:")
missing_ec = ecstat.isnull().sum()
if missing_ec.sum() > 0:
    display(missing_ec[missing_ec > 0])
    print(f"  Total: {missing_ec.sum()} missing values ({100*missing_ec.sum()/(ecstat.shape[0]*ecstat.shape[1]):.2f}% of all cells)")
else:
    print("  ✅ No missing values in ECstat!")

# ============================================================================
# PART 2: Detailed Analysis (if missing values exist)
# ============================================================================
has_missing = (missing_cl.sum() > 0) or (missing_ec.sum() > 0)

if has_missing:
    print("\n" + "="*80)
    print("PART 2: Detailed Missing Value Analysis")
    print("="*80)
    
    # Analyze CLstat if it has missing values
    if missing_cl.sum() > 0:
        print("\n🔍 CLstat - Rows with missing values:")
        missing_rows = clstat[clstat.isnull().any(axis=1)]
        print(f"  Total rows affected: {len(missing_rows):,} ({100*len(missing_rows)/len(clstat):.2f}% of dataset)")
        print("\n  Sample of rows with missing values:")
        display(missing_rows.head(10))
        
        # Check if missing is systematic or random
        print("\n📈 Temporal pattern of missing values in CLstat:")
        clstat_temp = clstat.copy()
        clstat_temp['has_missing'] = clstat_temp.isnull().any(axis=1)
        clstat_temp['Time_parsed'] = pd.to_datetime(clstat_temp['Time'], errors='coerce')
        
        if clstat_temp['Time_parsed'].notna().any():
            clstat_temp['hour'] = clstat_temp['Time_parsed'].dt.hour
            clstat_temp['date'] = clstat_temp['Time_parsed'].dt.date
            
            # Missing by hour
            missing_by_hour = clstat_temp.groupby('hour')['has_missing'].sum()
            print("\n  Missing values by hour of day:")
            display(missing_by_hour[missing_by_hour > 0])
            
            # Missing by date
            missing_by_date = clstat_temp.groupby('date')['has_missing'].sum().sort_values(ascending=False)
            if len(missing_by_date) > 0:
                print("\n  Top 10 dates with most missing values:")
                display(missing_by_date.head(10))
        
        # Check if specific base stations/cells have more missing values
        print("\n🏢 Missing values by Base Station/Cell:")
        missing_by_bs = clstat_temp.groupby('BS')['has_missing'].sum().sort_values(ascending=False)
        if len(missing_by_bs[missing_by_bs > 0]) > 0:
            print(f"  Base stations with missing values: {(missing_by_bs > 0).sum()}")
            print("\n  Top 10 base stations with most missing values:")
            display(missing_by_bs.head(10))
    
    # Analyze ECstat if it has missing values
    if missing_ec.sum() > 0:
        print("\n🔍 ECstat - Rows with missing values:")
        missing_rows_ec = ecstat[ecstat.isnull().any(axis=1)]
        print(f"  Total rows affected: {len(missing_rows_ec):,} ({100*len(missing_rows_ec)/len(ecstat):.2f}% of dataset)")
        print("\n  Sample of rows with missing values:")
        display(missing_rows_ec.head(10))

# ============================================================================
# PART 3: Recommendations
# ============================================================================
print("\n" + "="*80)
print("PART 3: Missing Value Strategy Recommendation")
print("="*80)

total_missing_cl = missing_cl.sum()
total_missing_ec = missing_ec.sum()
total_missing = total_missing_cl + total_missing_ec

if total_missing == 0:
    print("\n✅ **RECOMMENDATION: SKIP MISSING VALUE HANDLING**")
    print("\n  Reason: No missing values detected in raw data!")
    print("  Action: Remove ffill/bfill lines from cleaning code.")
    print("  Note: The only NaN values will come from lagged features later.")
    
elif total_missing_cl / (clstat.shape[0] * clstat.shape[1]) < 0.01 and total_missing_ec / (ecstat.shape[0] * ecstat.shape[1]) < 0.01:
    print("\n✅ **RECOMMENDATION: SIMPLE FORWARD/BACKWARD FILL (ffill/bfill)**")
    print("\n  Reason: < 1% missing values, likely random gaps")
    print("  Current approach is acceptable.")
    print("  Missing values:", total_missing)
    
elif total_missing_cl / (clstat.shape[0] * clstat.shape[1]) < 0.05:
    print("\n⚠️  **RECOMMENDATION: TIME-AWARE INTERPOLATION**")
    print("\n  Reason: 1-5% missing values")
    print("  Better approach: Use time-based interpolation with max gap limit")
    print("  Code suggestion:")
    print("    clstat['load'] = clstat.groupby(['BS', 'CellName'])['load'].apply(")
    print("        lambda x: x.interpolate(method='linear', limit=3)")
    print("    )")
    
else:
    print("\n❌ **RECOMMENDATION: INVESTIGATE DATA QUALITY ISSUES**")
    print("\n  Reason: > 5% missing values - significant data quality problem")
    print("  Action: Investigate why so much data is missing before filling")
    print("  Consider: Drop problematic base stations or time periods")

print("\n" + "="*80)
print("✓ Missing value analysis complete!")
print("  Review the output above before proceeding to Cell 7 (Data Cleaning)")
print("="*80)

---
## 7. Data Cleaning

### Steps:
1. Convert time columns to datetime
2. Check for invalid values (negative energy, load > 1, etc.)
3. Sort data by time (needed for feature engineering)
4. Verify energy saving mode columns are present

**⚠️ IMPORTANT:** Review Cell 6 output before running this cell. Based on the analysis, no missing value handling is needed for raw data.

In [ ]:
print("="*80)
print("DATA CLEANING")
print("="*80)

# Convert time columns to datetime
print("\n⏰ Converting time columns to datetime...")
clstat['Time'] = pd.to_datetime(clstat['Time'])
ecstat['Time'] = pd.to_datetime(ecstat['Time'])
print("  ✓ Time columns converted")

# Check for invalid values
print("\n🔍 Checking for invalid values...")
issues = []
if (bsinfo['TXpower'] < 0).any():
    issues.append(f"  ⚠️  {(bsinfo['TXpower'] < 0).sum()} negative TXpower values in BSinfo")
if (clstat['load'] < 0).any() or (clstat['load'] > 1).any():
    issues.append(f"  ⚠️  {((clstat['load'] < 0) | (clstat['load'] > 1)).sum()} invalid load values in CLstat")
if (ecstat['Energy'] < 0).any():
    issues.append(f"  ⚠️  {(ecstat['Energy'] < 0).sum()} negative energy values in ECstat")

if issues:
    for issue in issues:
        print(issue)
else:
    print("  ✓ No invalid values found")

# Sort data by time (needed for feature engineering later)
print("\n📋 Sorting data by time...")
clstat = clstat.sort_values(['BS', 'CellName', 'Time'])
ecstat = ecstat.sort_values(['BS', 'Time'])
print("  ✓ Data sorted")
print(f"    CLstat: {len(clstat):,} rows")
print(f"    ECstat: {len(ecstat):,} rows")

# Keep energy saving mode columns as separate features
print("\n⚡ Energy saving mode columns...")
esmode_cols = [col for col in clstat.columns if col.startswith('ESMode')]
if esmode_cols:
    print(f"  ✓ Keeping {len(esmode_cols)} ESMode columns as separate features: {esmode_cols}")
else:
    print("  ℹ️  No energy saving mode columns found")

print("\n✓ Data cleaning complete!")
print(f"  CLstat: {len(clstat):,} rows × {clstat.shape[1]} columns")
print(f"  ECstat: {len(ecstat):,} rows × {ecstat.shape[1]} columns")
print("\n  Note: No missing value handling needed (confirmed by Cell 6)")

---
## 8. Feature Engineering: Time-Based Features

Extract temporal patterns: hour of day, day of week, weekend indicator, peak hours, night time.

**Note:** `day_of_month` is excluded because the dataset only covers 7 days, which is insufficient to capture meaningful monthly patterns.

In [ ]:
print("="*80)
print("FEATURE ENGINEERING - Time-Based Features")
print("="*80)

# Time-based features for CLstat
print("\n⏰ Creating time-based features for CLstat...")
clstat['hour_of_day'] = clstat['Time'].dt.hour
clstat['day_of_week'] = clstat['Time'].dt.dayofweek
clstat['is_weekend'] = (clstat['day_of_week'] >= 5).astype(int)
clstat['is_peak_hour'] = clstat['hour_of_day'].apply(
    lambda x: 1 if (8 <= x <= 10) or (17 <= x <= 19) else 0
)
clstat['is_night_time'] = clstat['hour_of_day'].apply(
    lambda x: 1 if (22 <= x <= 23) or (0 <= x <= 6) else 0
)
print("  ✓ Created: hour_of_day, day_of_week, is_weekend, is_peak_hour, is_night_time")

# Time-based features for ECstat
print("\n⏰ Creating time-based features for ECstat...")
ecstat['hour_of_day'] = ecstat['Time'].dt.hour
ecstat['day_of_week'] = ecstat['Time'].dt.dayofweek
ecstat['is_weekend'] = (ecstat['day_of_week'] >= 5).astype(int)
ecstat['is_peak_hour'] = ecstat['hour_of_day'].apply(
    lambda x: 1 if (8 <= x <= 10) or (17 <= x <= 19) else 0
)
ecstat['is_night_time'] = ecstat['hour_of_day'].apply(
    lambda x: 1 if (22 <= x <= 23) or (0 <= x <= 6) else 0
)
print("  ✓ Created: hour_of_day, day_of_week, is_weekend, is_peak_hour, is_night_time")

print("\n✓ Time-based features complete!")
print(f"  CLstat: {clstat.shape[1]} total columns")
print(f"  ECstat: {ecstat.shape[1]} total columns")
print("\n  Note: day_of_month excluded (only 7 days of data - not meaningful for monthly patterns)")

In [ ]:
clstat

---
## 9. Feature Engineering: Lagged Features

Create features using previous time steps (t-1 and t-24 hours) for load and energy.

In [ ]:
print("="*80)
print("FEATURE ENGINEERING - Lagged Features")
print("="*80)

# Lagged features for load
print("\n📊 Creating lagged features for load...")
clstat = clstat.sort_values(['BS', 'CellName', 'Time'])
clstat['load_lag1'] = clstat.groupby(['BS', 'CellName'])['load'].shift(1)
clstat['load_lag24'] = clstat.groupby(['BS', 'CellName'])['load'].shift(24)
print("  ✓ Created: load_lag1 (t-1 hour), load_lag24 (t-24 hours)")
print(f"    NaN values in load_lag1: {clstat['load_lag1'].isnull().sum():,}")
print(f"    NaN values in load_lag24: {clstat['load_lag24'].isnull().sum():,}")

# Lagged features for energy
print("\n⚡ Creating lagged features for energy...")
ecstat = ecstat.sort_values(['BS', 'Time'])
ecstat['energy_lag1'] = ecstat.groupby('BS')['Energy'].shift(1)
ecstat['energy_lag24'] = ecstat.groupby('BS')['Energy'].shift(24)
print("  ✓ Created: energy_lag1 (t-1 hour), energy_lag24 (t-24 hours)")
print(f"    NaN values in energy_lag1: {ecstat['energy_lag1'].isnull().sum():,}")
print(f"    NaN values in energy_lag24: {ecstat['energy_lag24'].isnull().sum():,}")

print("\n✓ Lagged features complete!")
print(f"  CLstat: {clstat.shape[1]} total columns")
print(f"  ECstat: {ecstat.shape[1]} total columns")

---
## 10. Feature Engineering: Rolling Statistics

Create rolling mean and standard deviation features using 3-hour windows.

In [ ]:
print("="*80)
print("FEATURE ENGINEERING - Rolling Statistics")
print("="*80)

# Rolling statistics for load
print("\n📊 Creating rolling statistics for load (3-hour window)...")
clstat['load_rolling_mean_3h'] = clstat.groupby(['BS', 'CellName'])['load'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)
clstat['load_rolling_std_3h'] = clstat.groupby(['BS', 'CellName'])['load'].transform(
    lambda x: x.rolling(window=3, min_periods=1).std()
)
print("  ✓ Created: load_rolling_mean_3h, load_rolling_std_3h")

# Rolling statistics for energy
print("\n⚡ Creating rolling statistics for energy (3-hour window)...")
ecstat['energy_rolling_mean_3h'] = ecstat.groupby('BS')['Energy'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)
ecstat['energy_rolling_std_3h'] = ecstat.groupby('BS')['Energy'].transform(
    lambda x: x.rolling(window=3, min_periods=1).std()
)
print("  ✓ Created: energy_rolling_mean_3h, energy_rolling_std_3h")

print("\n✓ Rolling statistics complete!")
print(f"  CLstat: {clstat.shape[1]} total columns")
print(f"  ECstat: {ecstat.shape[1]} total columns")

---
## 11. Data Merging: Step 1 - CLstat + BSinfo

Merge cell-level statistics with base station metadata on `[BS, CellName]` keys.

In [ ]:
print("="*80)
print("DATA MERGING - Step 1: CLstat + BSinfo")
print("="*80)

print("\n🔗 Merging CLstat with BSinfo...")
print(f"  CLstat shape: {clstat.shape}")
print(f"  BSinfo shape: {bsinfo.shape}")
print(f"  Join keys: ['BS', 'CellName']")
print(f"  Join type: LEFT (preserve all CLstat records)")

df_merged = pd.merge(clstat, bsinfo, on=['BS', 'CellName'], how='left')

print(f"\n  ✓ Merge complete!")
print(f"    Result shape: {df_merged.shape}")
print(f"    Rows with missing values: {df_merged.isnull().any(axis=1).sum():,}")

# Show sample of merged data
print("\n📊 Sample of merged data:")
display(df_merged.head())

print("\n📋 Columns added from BSinfo:")
new_cols = [col for col in df_merged.columns if col not in clstat.columns]
print(f"  {new_cols}")

---
## 12. Data Merging: Step 2 - Result + ECstat

Merge the result with energy consumption data on `[Time, BS]` keys.

In [ ]:
print("="*80)
print("DATA MERGING - Step 2: Result + ECstat")
print("="*80)

print("\n🔗 Merging with ECstat...")
print(f"  Current shape: {df_merged.shape}")
print(f"  ECstat shape: {ecstat.shape}")
print(f"  Join keys: ['Time', 'BS']")
print(f"  Join type: LEFT (preserve all existing records)")

df_final = pd.merge(df_merged, ecstat, on=['Time', 'BS'], how='left', suffixes=('', '_ec'))

print(f"\n  ✓ Merge complete!")
print(f"    Result shape: {df_final.shape}")
print(f"    Rows with missing values: {df_final.isnull().any(axis=1).sum():,}")

# Check which columns have missing values
missing_cols = df_final.columns[df_final.isnull().any()].tolist()
if missing_cols:
    print(f"\n  ⚠️  Columns with missing values:")
    for col in missing_cols[:10]:  # Show first 10
        nan_count = df_final[col].isnull().sum()
        print(f"    - {col}: {nan_count:,} NaN ({100*nan_count/len(df_final):.1f}%)")
    if len(missing_cols) > 10:
        print(f"    ... and {len(missing_cols)-10} more columns")

# Handle duplicate columns
duplicate_cols = [col for col in df_final.columns if col.endswith('_ec') and col[:-3] in df_final.columns]
if duplicate_cols:
    print(f"\n🔧 Resolving {len(duplicate_cols)} duplicate columns...")
    for col in duplicate_cols:
        base_col = col[:-3]
        if base_col not in ['Energy']:  # Keep energy-related columns
            df_final[base_col] = df_final[base_col].fillna(df_final[col])
    df_final = df_final.drop(columns=duplicate_cols)
    print(f"  ✓ Resolved duplicate columns")

print(f"\n📊 Current shape: {df_final.shape}")

---
## 13. Final Cleanup: Remove Rows with Missing Values

Remove rows that have missing values after the merge (unmatched records and lag features).

---
## 14. Save Processed Data - Multiple Variants

We'll create **3 variants** for different modeling approaches:

1. **netop_ml_baseline.csv** - Raw features only (no time, no lag, no rolling)
2. **netop_ml_time.csv** - Raw + time features (no lag, no rolling)
3. **netop_ml_full.csv** - Everything (raw + time + lag + rolling)

This allows us to compare model performance with different feature sets and understand the value of feature engineering.

In [ ]:
print("="*80)
print("SAVING PROCESSED DATA - Multiple Variants")
print("="*80)

# =============================================================================
# Define feature sets for each variant
# =============================================================================

# Variant 1: Baseline - Raw features only
baseline_features = [
    'Time', 'BS', 'CellName', 'load',
    'ESMode1', 'ESMode2', 'ESMode3', 'ESMode4', 'ESMode5', 'ESMode6',
    'RUType', 'Mode', 'Frequency', 'Bandwidth', 'Antennas', 'TXpower',
    'Energy'
]

# Variant 2: Time - Raw + time features
time_features = baseline_features + [
    'hour_of_day', 'day_of_week', 'is_weekend', 'is_peak_hour', 'is_night_time'
]

# Variant 3: Full - Everything (raw + time + lag + rolling)
lag_rolling_features = [
    'load_lag1', 'load_lag24', 'energy_lag1', 'energy_lag24',
    'load_rolling_mean_3h', 'load_rolling_std_3h',
    'energy_rolling_mean_3h', 'energy_rolling_std_3h'
]
full_features = time_features + lag_rolling_features

# =============================================================================
# Verify all columns exist in df_final
# =============================================================================
print("\n🔍 Verifying column availability...")
missing_baseline = [col for col in baseline_features if col not in df_final.columns]
missing_time = [col for col in time_features if col not in df_final.columns]
missing_full = [col for col in full_features if col not in df_final.columns]

if missing_baseline:
    print(f"  ⚠️  Missing in baseline: {missing_baseline}")
if missing_time:
    print(f"  ⚠️  Missing in time: {missing_time}")
if missing_full:
    print(f"  ⚠️  Missing in full: {missing_full}")

if not (missing_baseline or missing_time or missing_full):
    print("  ✓ All required columns present!")

# =============================================================================
# Save Variant 1: Baseline (Raw features only)
# =============================================================================
print("\n💾 Saving Variant 1: netop_ml_baseline.csv...")
df_baseline = df_final[baseline_features].copy()
df_baseline.to_csv('processed_data/netop_ml_baseline.csv', index=False)
print(f"  ✓ Saved: {df_baseline.shape[0]:,} rows × {df_baseline.shape[1]} columns")
print(f"  Features: Raw features only (no time, no lag, no rolling)")

# =============================================================================
# Save Variant 2: Time (Raw + time features)
# =============================================================================
print("\n💾 Saving Variant 2: netop_ml_time.csv...")
df_time = df_final[time_features].copy()
df_time.to_csv('processed_data/netop_ml_time.csv', index=False)
print(f"  ✓ Saved: {df_time.shape[0]:,} rows × {df_time.shape[1]} columns")
print(f"  Features: Raw + time features (no lag, no rolling)")

# =============================================================================
# Save Variant 3: Full (Everything)
# =============================================================================
print("\n💾 Saving Variant 3: netop_ml_full.csv...")
df_full = df_final[full_features].copy()
df_full.to_csv('processed_data/netop_ml_full.csv', index=False)
print(f"  ✓ Saved: {df_full.shape[0]:,} rows × {df_full.shape[1]} columns")
print(f"  Features: Raw + time + lag + rolling features")

# =============================================================================
# Save individual processed files (for reference)
# =============================================================================
print("\n💾 Saving individual processed files (for reference)...")
clstat.to_csv('processed_data/clstat_processed.csv', index=False)
print(f"  ✓ Saved: processed_data/clstat_processed.csv ({clstat.shape[0]:,} rows)")

ecstat.to_csv('processed_data/ecstat_processed.csv', index=False)
print(f"  ✓ Saved: processed_data/ecstat_processed.csv ({ecstat.shape[0]:,} rows)")

# =============================================================================
# Summary
# =============================================================================
print("\n" + "="*80)
print("SUMMARY - Files Created")
print("="*80)
print(f"""
📁 processed_data/
  ├── netop_ml_baseline.csv    {df_baseline.shape[0]:>6,} rows × {df_baseline.shape[1]:>2} cols  (Raw only)
  ├── netop_ml_time.csv         {df_time.shape[0]:>6,} rows × {df_time.shape[1]:>2} cols  (Raw + Time)
  ├── netop_ml_full.csv         {df_full.shape[0]:>6,} rows × {df_full.shape[1]:>2} cols  (Full features)
  ├── clstat_processed.csv      {clstat.shape[0]:>6,} rows × {clstat.shape[1]:>2} cols  (Reference)
  └── ecstat_processed.csv      {ecstat.shape[0]:>6,} rows × {ecstat.shape[1]:>2} cols  (Reference)
""")

print("✓ All files saved successfully!")
print("="*80)

In [ ]:
print("="*80)
print("SAVING PROCESSED DATA")
print("="*80)

# Save final merged dataset
print("\n💾 Saving netop_processed.csv...")
df_final.to_csv('processed_data/netop_processed.csv', index=False)
print(f"  ✓ Saved: processed_data/netop_processed.csv ({df_final.shape[0]:,} rows)")

# Save individual processed files
print("\n💾 Saving individual processed files...")
clstat.to_csv('processed_data/clstat_processed.csv', index=False)
print(f"  ✓ Saved: processed_data/clstat_processed.csv ({clstat.shape[0]:,} rows)")

ecstat.to_csv('processed_data/ecstat_processed.csv', index=False)
print(f"  ✓ Saved: processed_data/ecstat_processed.csv ({ecstat.shape[0]:,} rows)")

print("\n✓ All files saved successfully!")

---
## 15. Summary Report

Final statistics and summary of the preprocessing pipeline.

In [ ]:
print("="*80)
print("PREPROCESSING SUMMARY")
print("="*80)

print(f"\n📊 Final Dataset Shape: {df_final.shape[0]:,} rows × {df_final.shape[1]} columns")

print("\n📋 All Columns:")
for i, col in enumerate(df_final.columns, 1):
    print(f"  {i:2d}. {col}")

print("\n🎯 Data Quality:")
print(f"  - Missing values: {df_final.isnull().sum().sum()}")
print(f"  - Duplicate rows: {df_final.duplicated().sum()}")
print(f"  - Date range: {df_final['Time'].min()} to {df_final['Time'].max()}")
print(f"  - Number of base stations: {df_final['BS'].nunique()}")
print(f"  - Number of cells: {df_final['CellName'].nunique()}")
print(f"  - Total measurements: {len(df_final):,}")

print("\n📈 Key Statistics:")
display(df_final[['load', 'Energy', 'TXpower', 'Bandwidth']].describe())

# Show ESMode statistics if available
esmode_cols = [col for col in df_final.columns if col.startswith('ESMode')]
if esmode_cols:
    print("\n⚡ Energy Saving Mode Statistics:")
    print(f"  Found {len(esmode_cols)} ESMode columns: {esmode_cols}")
    for col in esmode_cols:
        active_pct = 100 * (df_final[col] > 0).mean()
        if active_pct > 0:
            print(f"  - {col}: Active {active_pct:.2f}% of the time")

print("\n" + "="*80)
print("✓ PREPROCESSING COMPLETE!")
print("="*80)

---
## 16. Quick Data Exploration (Optional)

Visualize some key patterns in the processed data.

In [ ]:
# Quick visualization of load and energy patterns
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Load distribution
axes[0, 0].hist(df_final['load'], bins=50, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Load Distribution', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Load')
axes[0, 0].set_ylabel('Frequency')

# Energy distribution
axes[0, 1].hist(df_final['Energy'], bins=50, color='lightcoral', edgecolor='black')
axes[0, 1].set_title('Energy Distribution', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Energy (W)')
axes[0, 1].set_ylabel('Frequency')

# Load by hour
hourly_load = df_final.groupby('hour_of_day')['load'].mean()
axes[1, 0].plot(hourly_load.index, hourly_load.values, marker='o', linewidth=2, color='green')
axes[1, 0].set_title('Average Load by Hour of Day', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Hour of Day')
axes[1, 0].set_ylabel('Average Load')
axes[1, 0].grid(True, alpha=0.3)

# Energy by hour
hourly_energy = df_final.groupby('hour_of_day')['Energy'].mean()
axes[1, 1].plot(hourly_energy.index, hourly_energy.values, marker='s', linewidth=2, color='orange')
axes[1, 1].set_title('Average Energy by Hour of Day', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Hour of Day')
axes[1, 1].set_ylabel('Average Energy (W)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("📊 Quick visualizations generated!")

In [ ]:
df_final.columns